In [1]:
import os
import torch
import pytorch_lightning as pl

# Import your specific model class
# Adjust the import path if your working directory is different
from source.modules.ipes_cnn import IpesCnn

In [ ]:
# TODO: Update these paths!
CHECKPOINT_PATH = r"C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\checkpoints\last.ckpt"  
OUTPUT_LIBTORCH_PATH = r"exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.pt"                            
YAML_CONFIG_PATH = r"C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\config.yaml"     

In [3]:
print(f"Loading config from {YAML_CONFIG_PATH}...")

with open(YAML_CONFIG_PATH, 'r') as file:
    import yaml
    config = yaml.safe_load(file)
    
model_kwargs = config['model']['init_args']

# 2. Extract the "linked" arguments that live in the data section
data_kwargs = config['data']['init_args']
model_kwargs['pts_to_img_methods'] = data_kwargs['pts_to_img_methods']
model_kwargs['hm_size'] = data_kwargs['hm_size']
model_kwargs['in_file'] = data_kwargs['in_file']

# The 'workers' arg is null in the model block, so we grab it from data
if model_kwargs.get('workers') is None:
    model_kwargs['workers'] = data_kwargs.get('workers', 0)


Loading config from C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\config.yaml...


In [4]:

print(f"Loading model from {CHECKPOINT_PATH}...")

# Load the PyTorch Lightning module from the checkpoint
network = IpesCnn.load_from_checkpoint(
    checkpoint_path=CHECKPOINT_PATH,
    **model_kwargs
)

# This disables dropout and locks BatchNorm statistics before tracing
network.eval()

print("Model loaded and set to eval mode successfully!")

Loading model from C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\checkpoints\last.ckpt...
Model loaded and set to eval mode successfully!


In [ ]:
# THE CRITICAL CUDNN FIX FOR DYNAMIC BATCH SIZES IN C++
# This prevents TorchScript from hardcoding batch-specific kernel optimizations into the graph.

print("Applying cuDNN deterministic settings...")
torch.backends.cudnn.enabled = True 
torch.backends.cudnn.allow_tf32 = False  # seems bugged, caused CUDNN_STATUS_EXECUTION_FAILED on C++ side
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

Applying cuDNN deterministic settings...


In [13]:
def make_example_data(net, batch_size=1):
    """
    Generates dummy data matching the exact shapes the model expects.
    Batch size is strictly 1 to ensure a dynamic, flexible graph.
    """
    res = net.hm_interp_size
    example_inputs = dict()

    for m in net.input_methods:
        example_inputs[f'patch_hm_{m}'] = torch.rand(batch_size, 1, res, res)
        example_inputs[f'patch_rgb_{m}'] = torch.rand(batch_size, 3, res, res)
        
    example_inputs['patch_hm_mask'] = torch.rand(batch_size, 1, res, res)
    
    return example_inputs

# Generate the dummy dictionary
dummy_inputs = make_example_data(network, batch_size=1)

print("Dummy inputs generated:")
for k, v in dummy_inputs.items():
    print(f"  {k}: {v.shape}")

Dummy inputs generated:
  patch_hm_nearest: torch.Size([1, 1, 96, 96])
  patch_rgb_nearest: torch.Size([1, 3, 96, 96])
  patch_hm_linear: torch.Size([1, 1, 96, 96])
  patch_rgb_linear: torch.Size([1, 3, 96, 96])
  patch_hm_mask: torch.Size([1, 1, 96, 96])
  patch_rgb_mask: torch.Size([1, 3, 96, 96])


In [16]:
print(f"\nTracing model to {OUTPUT_LIBTORCH_PATH}")

# Ensure the output directory exists
os.makedirs(os.path.dirname(OUTPUT_LIBTORCH_PATH), exist_ok=True)

# Use PyTorch Lightning's built-in wrapper to trace and save
network.to_torchscript(
    file_path=OUTPUT_LIBTORCH_PATH, 
    method='trace', 
    example_inputs=dummy_inputs
)

# Fails with 1000 errors
# from torch.utils.mobile_optimizer import optimize_for_mobile # (Works for desktop LibTorch too)
# traced_model = network.to_torchscript(method='trace', example_inputs=dummy_inputs)
# # Freeze the model (removes gradients, fuses BatchNorms)
# frozen_model = torch.jit.freeze(traced_model)
# # Optimize for inference
# optimized_model = torch.jit.optimize_for_inference(frozen_model)
# # Save the optimized version
# optimized_model.save(OUTPUT_LIBTORCH_PATH)

print(f"Tracing complete! Model saved to {OUTPUT_LIBTORCH_PATH} and is ready for C++ inference.")


Tracing model to exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.pt
Tracing complete! Model saved to exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.pt and is ready for C++ inference.
